In [ ]:
!pip install -U sacremoses datasets transformers accelerate seaborn wandb

In [ ]:
import torch
import numpy as np
import pandas as pd

RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [ ]:
data = pd.read_csv("data/hate_train.csv")
data

In [ ]:
data["label"] = data["label"].astype(bool)
data

In [ ]:
data["label"].value_counts().plot.bar(
    title="Label Counts", xlabel="Label", ylabel="Count"
)


In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    data, test_size=0.2, random_state=RANDOM_SEED, stratify=data["label"]
)
train_data["label"].value_counts().plot.bar(
    title="Label Counts", xlabel="Label", ylabel="Count"
)


In [ ]:
sentences = []
with open("data/hate_test_data.txt", "r") as f:
    for line in f:
        sentences.append(line.strip())

sntences_df = pd.DataFrame(sentences, columns=["sentence"])
sntences_df

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(sntences_df)

print(train_dataset[0])
print(val_dataset[0])
print(test_dataset[0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "deepsense-ai/trelbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, problem_type="single_label_classification"
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"], padding="max_length", truncation=True, max_length=256
    )


tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)
tokenized_train_dataset[0]

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Calculated weights: {class_weights}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
from torch import nn
from transformers import Trainer


class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
import wandb

wandb.login()

training_args = TrainingArguments(
    output_dir=MODEL_NAME + "-hate-speech-classification",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=128,
    per_device_train_batch_size=128,
    num_train_epochs=20,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
val_predictions = trainer.predict(tokenized_val_dataset)

y_preds = np.argmax(val_predictions.predictions, axis=1)

y_true = val_predictions.label_ids

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


def plot_confusion_matrix(y_true, y_preds):
    cm = confusion_matrix(y_true, y_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
    )

    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:
plot_confusion_matrix(y_true, y_preds)

In [ ]:
test_predictions = trainer.predict(tokenized_test_dataset)

In [ ]:
y_preds = np.argmax(test_predictions.predictions, axis=1)

y_preds_series = pd.Series(y_preds)
y_preds_series.to_csv("pred.csv", index=False, header=False)

In [ ]:
y_preds_series